# Homework 5 — Deploying Machine Learning Models

For the API and Docker questions, use the terminal commands below. The course supplies the pretrained model; we do not retrain it.

## Set up the project folder

The Python cells below work when the notebook's current working directory is either `05-homework` or the `ml-zoomcamp` repository root.

In [1]:
from pathlib import Path
import sys

project = Path.cwd()
if not (project / "pipeline.bin").exists():
    project = project / "05-deployment" / "05-homework"
assert (project / "pipeline.bin").exists(), "Run the notebook from the homework folder or repository root"
print("Homework directory:", project.resolve())


print(sys.version)
print(sys.executable)

Homework directory: C:\Users\iTC\Documents\ml-zoomcamp\05-deployment\05-homework
3.11.15 (main, Aug  7 2026, 02:24:19) [MSC v.1944 64 bit (AMD64)]
c:\Users\iTC\Documents\ml-zoomcamp\05-deployment\05-homework\.venv\Scripts\python.exe


## Q1 — Environment check


In [2]:
import subprocess

print(subprocess.check_output(["uv", "--version"], text=True).strip())

uv 0.13.4 (7456b60bc 2026-10-20 x86_64-pc-windows-msvc)


## Q2 — Locked scikit-learn version


In [3]:
import tomllib

with (project / "pyproject.toml").open("rb") as f:
    config = tomllib.load(f)
with (project / "uv.lock").open("rb") as f:
    lock = tomllib.load(f)

version = next(p["version"] for p in lock["package"] if p["name"] == "scikit-learn")
print("scikit-learn:", version)
print("Declared:", next(d for d in config["project"]["dependencies"] if d.startswith("scikit-learn")))

scikit-learn: 1.7.2
Declared: scikit-learn==1.7.2


## Q3 — Load the model and predict

`pipeline.bin` contains the fitted `DictVectorizer` and `LogisticRegression` pipeline. Load it with `pickle` and predict for the **Q3** record.

In [4]:
import pickle
import sys

sys.path.insert(0, str(project.resolve()))
from model import normalize_record

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with (project / "pipeline.bin").open("rb") as f:
    pipeline = pickle.load(f)

probability = pipeline.predict_proba([normalize_record(lead)])[0, 1]
print(f"Q3: {probability:.3f}")

Q3: 0.533


## Q4 — Serve the model with FastAPI

**Start the FastAPI server and test the prediction endpoint in two terminals**. In the first one, go to `05-deployment/05-homework` and start the provided API:

```powershell
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

In the second terminal, from the **same directory**, send the Q4 example with the provided client:

```powershell
uv run python q6_test.py
Invoke-RestMethod http://localhost:9696/health
```

Expected Q4 probability: **0.770** (`0.769799` before rounding). The API responds with a probability and a boolean `conversion` field. Stop the server with Ctrl+C after testing.

## Q5 — Python Docker image

Read the first `FROM` instruction in the supplied `Dockerfile`. The answer is **`python:3.11.15-slim-bookworm`**. The Dockerfile also pins the exact image digest.

## Q6 — Make the same prediction in Docker

Stop the local API first to free port 9696. From the homework directory, run in terminal 1:

```powershell
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

In terminal 2, from the homework directory:

```powershell
uv run python q6_test.py
Invoke-RestMethod http://localhost:9696/health
```

Expected Q6 probability: **0.770**, the same as Q4. Stop the container with Ctrl+C.

## Answers

| Question | Answer |
| --- | --- |
| Q1 | Local `uv` version, `0.13.4`|
| Q2 | `1.7.2` |
| Q3 | `0.533` |
| Q4 | `0.770` |
| Q5 | `python:3.11.15-slim-bookworm` |
| Q6 | `0.770` |
